## 06. Model Evaluation

In [5]:
from pathlib import Path
import time
import joblib
import sys

sys.path.append(str(Path.cwd().parents[0]))
from src.validation import *
from src.tuning import show_metrics

In [10]:
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.ensemble import RandomForestClassifier
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# load the data
df = pd.read_csv('../data/processed/modeling_data.csv')
df

,dedup_group_id,text,has_url,has_money_mention,has_phone_in_text,exclamation_count,uppercase_ratio,has_otp,urgency,has_leetspeaks,typo_squatting,has_url_shortened,punctuation_count,word_count,text_count,target
0,0,papalduhin kita moneytoken ngayon basta bahala...,0,1,0,0,0.073171,0,0,0,0,0,6,32,164,1
1,1,boss panalunin kita pinata win moneytoken haba...,0,1,0,0,0.016484,0,0,1,0,0,11,35,182,1
2,2,okay makipag deal sayo cash ipapanalo kita mon...,0,1,0,0,0.020942,0,0,1,0,0,10,35,191,1
3,3,fex parcel p iisddcddtba reach please contact ...,0,0,1,0,0.119266,0,0,0,0,0,5,16,109,1
4,4,smart reminds reward points expire today pleas...,1,0,0,0,0.043860,0,1,0,0,0,11,13,114,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2339,2274,please note photo proof delivery required,0,0,0,0,0.018182,0,0,0,0,0,1,10,55,0
2340,2275,thank choosing j express philippines stay safe...,0,0,0,1,0.070588,0,0,0,0,0,3,15,85,0
2341,2276,delivery p ioiesskreo rider joanna cod moneyto...,0,1,0,1,0.231884,0,0,0,0,0,1,7,66,1
2342,2277,delivery p ioiesskreo rider jeffersan cod mone...,0,1,0,1,0.222222,0,0,0,0,0,1,7,69,1


In [12]:
binary_cols = ['has_url', 'has_money_mention', 'has_phone_in_text', 'has_otp',
                 'urgency', 'has_leetspeaks', 'typo_squatting', 'has_url_shortened']

numericals = ['exclamation_count', 'text_count', 'word_count',
              'punctuation_count', 'uppercase_ratio']

x = df['text']
y = df['target']

# splitting and vectorization
vectorizer = TfidfVectorizer(ngram_range=(1, 5), min_df=2, max_df=0.5, sublinear_tf=True)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
scaler = MinMaxScaler(clip=True)
train_idx, test_idx = next(sgkf.split(x, y, groups=df['dedup_group_id']))

train_data = df.iloc[train_idx].reset_index(drop=True)
test_data = df.iloc[test_idx].reset_index(drop=True)

X_train = vectorizer.fit_transform(train_data['text'])
X_test = vectorizer.transform(test_data['text'])
y_train = train_data['target']
y_test = test_data['target']

num_train = scaler.fit_transform(train_data[numericals])
num_test  = scaler.transform(test_data[numericals])

bin_train = train_data[binary_cols].to_numpy(dtype=float)
bin_test  = test_data[binary_cols].to_numpy(dtype=float)

X_train = hstack([X_train, csr_matrix(bin_train), csr_matrix(num_train)]).tocsr()
X_test  = hstack([X_test,  csr_matrix(bin_test),  csr_matrix(num_test)]).tocsr()

# init both naive bayes model and using hyperparamters
mnb = MultinomialNB(alpha=0.1, fit_prior=True)
cnb = ComplementNB(alpha=0.1, norm=False)

mnb.fit(X_train, y_train)
cnb.fit(X_train, y_train)

ypred_mnb = mnb.predict(X_test)
ypred_cnb = cnb.predict(X_test)

yprob_mnb = mnb.predict_proba(X_test)
yprob_cnb = cnb.predict_proba(X_test)

rf = RandomForestClassifier(
    class_weight=None, criterion='entropy', max_depth=200,
    max_features='log2', min_samples_leaf=1, min_samples_split=6, n_estimators=200,
    random_state=42  
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)

print(f"Training Data Size: {len(train_data)} | Test Data Size: {len(test_data)}")

Training Data Size: 1875 | Test Data Size: 469


In [13]:
# helper function for checking predictions
def prediction_check(model: BaseEstimator, data: pd.DataFrame) -> pd.DataFrame:

    texts = data['text'].values
    predicted = model.predict(texts)
    actuals = data['target'].values
    matched = (actuals == predicted).astype('int')

    pred_check = pd.DataFrame({
        'Text' :  texts,
        'Actual' : actuals,
        'Predicted' : predicted,
        "Is Matched" : matched
    })

    return pred_check

## Model Evaluation for CNB and MNB

In [14]:
names = build_feature_names(vectorizer, binary_cols, numericals)
stats = term_stats(X_train, y_train, names)

nb_tables = {
    'MNB': nb_word_table(mnb, feature_names=names, stats=stats, top_n=15, max_ngram=1),
    'CNB': nb_word_table(cnb, feature_names=names, stats=stats, top_n=15, max_ngram=1),
}

nb_tables['MNB'].head(10)

,feature,rank,kind,docs_Smishing,mean_Smishing,docs_Legitimate,mean_Legitimate,lean,score,class
0,bonus,1,word,4,0.000259,177,0.032593,Legitimate,4.512383,Legitimate
1,pagcor,2,word,0,0.000000,39,0.008280,Legitimate,4.510140,Legitimate
2,nanalo,3,word,0,0.000000,24,0.005218,Legitimate,4.057625,Legitimate
3,vip,4,word,0,0.000000,16,0.005115,Legitimate,4.038080,Legitimate
4,gojackpot,5,word,0,0.000000,16,0.004805,Legitimate,3.977203,Legitimate
5,libreng,6,word,1,0.000113,33,0.010006,Legitimate,3.881440,Legitimate
6,https,7,word,0,0.000000,17,0.004316,Legitimate,3.872826,Legitimate
7,magdeposito,8,word,0,0.000000,12,0.004156,Legitimate,3.836202,Legitimate
8,luck,9,word,0,0.000000,12,0.003837,Legitimate,3.758836,Legitimate
9,friday,10,word,0,0.000000,13,0.003835,Legitimate,3.758398,Legitimate


In [10]:
nb_tables['CNB'].head(10)

,feature,rank,kind,docs_Smishing,mean_Smishing,docs_Legitimate,mean_Legitimate,lean,score,class
0,redeem,1,word,0,0.0,22,0.012612,Legitimate,0.000139,Legitimate
1,gojackpot,2,word,0,0.0,24,0.012481,Legitimate,0.000139,Legitimate
2,points,3,word,0,0.0,20,0.010899,Legitimate,0.000133,Legitimate
3,reminds,4,word,0,0.0,18,0.009625,Legitimate,0.000128,Legitimate
4,epicwin,5,word,0,0.0,19,0.009414,Legitimate,0.000127,Legitimate
5,login,6,word,0,0.0,12,0.009400,Legitimate,0.000127,Legitimate
6,trusted,7,word,0,0.0,19,0.008836,Legitimate,0.000125,Legitimate
7,tangkilikin,8,word,0,0.0,15,0.007871,Legitimate,0.000120,Legitimate
8,soon,9,word,0,0.0,13,0.007684,Legitimate,0.000119,Legitimate
9,smart,10,word,0,0.0,12,0.007489,Legitimate,0.000118,Legitimate


In [ ]:
i = 3                                           
print(test_data['text'][i], '| actual:', TARGET_NAMES[y_test[i]])
explain_message_nb(cnb, X_test[i], names, top_n=10)

bigwln nanalo prize moneytoken magparehistro makakuha moneytoken maglaro moneytoken makakuha moneytoken libreng moneytoken unang deposito gdubw social | actual: Legitimate


,feature,value,contribution,toward
0,makakuha moneytoken,0.382337,-0.000042,Smishing
1,magparehistro,0.211028,-0.000037,Smishing
2,magparehistro makakuha,0.252912,-0.000034,Smishing
3,moneytoken makakuha,0.266842,-0.000033,Smishing
4,moneytoken makakuha moneytoken,0.266842,-0.000033,Smishing
5,prize,0.259378,0.000027,Legitimate
6,libreng moneytoken,0.233279,-0.000025,Smishing
7,prize moneytoken,0.286475,0.000022,Legitimate
8,libreng,0.187323,-0.000015,Smishing
9,deposito,0.192252,0.000011,Legitimate


In [15]:
i = 3                                           
print(test_data['text'][i], '| actual:', TARGET_NAMES[y_test[i]])
explain_message_nb(mnb, X_test[i], names, top_n=10)

globe reminds reward points expire today please visit redeem favorite gift urltoken | actual: Smishing


,feature,value,contribution,toward
0,has_url,1.000000,0.594586,Legitimate
1,expire today please visit,0.151474,0.543195,Legitimate
2,points expire today please visit,0.151474,0.543195,Legitimate
3,today please visit,0.151474,0.543195,Legitimate
4,visit redeem,0.154798,0.528078,Legitimate
5,expire today please visit redeem,0.154798,0.528078,Legitimate
6,please visit redeem,0.154798,0.528078,Legitimate
7,today please visit redeem,0.154798,0.528078,Legitimate
8,gift urltoken,0.163035,0.512246,Legitimate
9,visit redeem favorite,0.163035,0.509089,Legitimate


## Random Forest Classifier - Fetaure Importance

In [16]:
rf_imp  = rf_importance_table(rf, feature_names=names, stats=stats, top_n=15, max_ngram=1)
rf_perm = rf_permutation_table(rf, X_test, y_test, feature_names=names,
                               stats=stats, top_n=15, max_ngram=1)
rf_perm.head(10)

,feature,rank,kind,impurity_importance,drop_macro_f1,drop_macro_f1_std,drop_f1_Smishing,drop_f1_Legitimate,docs_Smishing,mean_Smishing,docs_Legitimate,mean_Legitimate,lean,class
0,libreng,1,word,0.002922,0.002245,0.000000,0.001730,0.002761,1,0.000113,33,0.010006,Legitimate,Legitimate
1,click,2,word,0.001197,0.002245,0.000000,0.001730,0.002761,51,0.005436,45,0.009746,Legitimate,Legitimate
2,de,3,word,0.001091,0.002245,0.000000,0.001730,0.002761,5,0.000758,41,0.006527,Legitimate,Legitimate
3,has_leetspeaks,4,word,0.004229,0.002229,0.001394,0.001760,0.002698,105,0.094255,187,0.245729,Legitimate,Legitimate
4,urltoken,5,word,0.010010,0.001765,0.002586,0.001443,0.002086,321,0.017949,401,0.038558,Legitimate,Legitimate
5,life,6,word,0.001277,0.000866,0.001060,0.000752,0.000979,0,0.000000,15,0.003828,Legitimate,Legitimate
6,sign,7,word,0.002261,0.000496,0.001647,0.000260,0.000732,9,0.000792,37,0.009217,Legitimate,Legitimate
7,bonus,8,word,0.016055,0.000480,0.002559,0.000290,0.000669,4,0.000259,177,0.032593,Legitimate,Legitimate
8,bid,9,word,0.001150,0.000433,0.000866,0.000376,0.000490,2,0.000538,23,0.005321,Legitimate,Legitimate
9,tw,10,word,0.002235,0.000433,0.000866,0.000376,0.000490,2,0.000184,38,0.008753,Legitimate,Legitimate


## Model Evaluation - Agreements across Models

In [17]:
consensus_words(nb_word_table(cnb, feature_names=names, stats=stats, top_n=30, max_ngram=1),
                rf_importance_table(rf, feature_names=names, stats=stats, top_n=30, max_ngram=1))

{'Legitimate': ['bonus', 'libreng', 'life', 'pagcor', 'tw'],
 'Smishing': ['advisories',
  'borrow',
  'calls',
  'data',
  'gb',
  'gigalife',
  'load',
  'loaded',
  'promos',
  'ref',
  'texts',
  'unli']}

## Predictions and Confident Mistakes

In [18]:
pt = {n: prediction_table(m, X_test, y_test, test_data['text'])
      for n, m in [('MNB', mnb), ('CNB', cnb), ('RF', rf)]}

pt['RF'][pt['RF'].correct == 0].sort_values('confidence', ascending=False)

,text,actual,predicted,p_Smishing,p_Legitimate,confidence,correct
1,globe reminds reward points expire today pleas...,Smishing,Legitimate,0.000000,1.000000,1.000000,0
2,globe reminds reward points expire today pleas...,Smishing,Legitimate,0.000000,1.000000,1.000000,0
3,globe reminds reward points expire today pleas...,Smishing,Legitimate,0.000000,1.000000,1.000000,0
4,globe reminds reward points expire today pleas...,Smishing,Legitimate,0.000000,1.000000,1.000000,0
461,omg asourcepan,Legitimate,Smishing,0.762201,0.237799,0.762201,0
457,daily deposit reward back deposit moneytoken g...,Smishing,Legitimate,0.262741,0.737259,0.737259,0
448,april bill summary sent email view estatement ...,Legitimate,Smishing,0.686360,0.313640,0.686360,0
81,wi games official partner gcash one platform a...,Legitimate,Smishing,0.643958,0.356042,0.643958,0
343,facebook account recently accessed instagram r...,Smishing,Legitimate,0.378319,0.621681,0.621681,0
439,metacash pro verification code valid minutes m...,Legitimate,Smishing,0.614880,0.385120,0.614880,0


## Bundling Vectorizer and Models